In [ ]:
!git clone https://github.com/DepthAnything/Depth-Anything-V2.git
%cd /content/Depth-Anything-V2
!pip install -q -r metric_depth/requirements.txt ultralytics
print("Done")

In [ ]:
import os, urllib.request

CHECKPOINT_DIR = "/content/Depth-Anything-V2/metric_depth/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
CHECKPOINT_PATH = os.path.join(CHECKPOINT_DIR, "depth_anything_v2_metric_hypersim_vits.pth")
URL = "https://huggingface.co/depth-anything/Depth-Anything-V2-Metric-Hypersim-Small/resolve/main/depth_anything_v2_metric_hypersim_vits.pth"

if not os.path.exists(CHECKPOINT_PATH):
    urllib.request.urlretrieve(URL, CHECKPOINT_PATH)

print("Size MB:", round(os.path.getsize(CHECKPOINT_PATH) / (1024 * 1024), 2))

In [ ]:
import sys, importlib, torch

metric_path = "/content/Depth-Anything-V2/metric_depth"
if metric_path not in sys.path:
    sys.path.insert(0, metric_path)
importlib.invalidate_caches()

from depth_anything_v2.dpt import DepthAnythingV2

depth_model = DepthAnythingV2(
    encoder="vits", features=64,
    out_channels=[48, 96, 192, 384], max_depth=20
)
depth_model.load_state_dict(
    torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=True),
    strict=True
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
depth_model = depth_model.to(device).eval()
print("Model loaded on", device)

In [ ]:
from ultralytics import YOLO

yolo_model = YOLO("yolo26n.pt")

print("YOLO model loaded successfully!")

In [ ]:

def detect_objects(image_bgr, confidence=0.25):

    if image_bgr is None:
        raise ValueError("Input image is empty.")

    results = yolo_model.predict(
        source=image_bgr,
        conf=confidence,
        verbose=False
    )

    detections = []

    for result in results:
        for box in result.boxes:

            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()

            class_id = int(box.cls[0])
            label = result.names[class_id]
            score = float(box.conf[0])

            bbox = (
                int(x1),
                int(y1),
                int(x2),
                int(y2)
            )

            detections.append({
                "object": label,
                "confidence": score,
                "bbox": bbox
            })

    return detections

print("Object detection function ready!")


In [ ]:

import numpy as np

INPUT_SIZE = 518
INSET_RATIO = 0.20

def generate_depth_map(image_bgr):
    if image_bgr is None:
        raise ValueError("Image is empty.")

    depth = depth_model.infer_image(image_bgr, INPUT_SIZE)
    return np.asarray(depth, dtype=np.float32)

print("Depth model is ready")


In [ ]:

def calculate_median_depth(depth_map, bbox, inset_ratio=INSET_RATIO):

    height, width = depth_map.shape[:2]

    x1, y1, x2, y2 = map(int, bbox)

    x1 = max(0, min(x1, width))
    x2 = max(0, min(x2, width))
    y1 = max(0, min(y1, height))
    y2 = max(0, min(y2, height))

    if x2 <= x1 or y2 <= y1:
        raise ValueError("Invalid bounding box.")

    inset_x = int((x2 - x1) * inset_ratio)
    inset_y = int((y2 - y1) * inset_ratio)

    region = depth_map[
        y1 + inset_y:y2 - inset_y,
        x1 + inset_x:x2 - inset_x
    ]

    if region.size == 0:
        region = depth_map[y1:y2, x1:x2]

    valid = region[np.isfinite(region) & (region > 0)]

    if valid.size == 0:
        raise ValueError("No valid depth values found.")

    return float(np.median(valid))

print("Median depth function ready")


In [ ]:

SCALE_FACTOR = 1.05
OFFSET = -0.10

def calibrate_distance(raw_depth, scale=SCALE_FACTOR, offset=OFFSET):

    calibrated_distance = (scale * raw_depth) + offset

    return float(max(0.0, calibrated_distance))

print("Calibration loaded successfully!")
print("Scale Factor:", SCALE_FACTOR)
print("Offset:", OFFSET)


In [ ]:

def fit_calibration(raw_depths, actual_distances):
    """
    Calculate calibration slope and intercept
    from raw depth values and actual measured distances.
    """

    raw_depths = np.asarray(raw_depths, dtype=np.float64)
    actual_distances = np.asarray(actual_distances, dtype=np.float64)

    if len(raw_depths) != len(actual_distances):
        raise ValueError("Both lists must have the same length.")

    if len(raw_depths) < 2:
        raise ValueError("At least two calibration samples are required.")

    if not np.all(np.isfinite(raw_depths)) or not np.all(np.isfinite(actual_distances)):
        raise ValueError("Calibration values must be finite.")

    slope, intercept = np.polyfit(
        raw_depths,
        actual_distances,
        1
    )

    print("Slope:", slope)
    print("Intercept:", intercept)

    return float(slope), float(intercept)


print("Calibration fitting function is ready!")


In [ ]:

def estimate_distance(depth_map, bbox, scale, offset):
    raw_depth = calculate_median_depth(depth_map, bbox)

    calibrated = calibrate_distance(
        raw_depth,
        scale,
        offset
    )

    return {
        "raw_depth_m": raw_depth,
        "calibrated_distance_m": calibrated
    }


def process_detection(image_bgr, bbox, scale, offset):
    depth_map = generate_depth_map(image_bgr)

    return estimate_distance(
        depth_map,
        bbox,
        scale,
        offset
    )


print("Distance estimation pipeline is ready!")


In [ ]:

def detect_objects_with_distance(image_bgr):

    detections = detect_objects(image_bgr)

    depth_map = generate_depth_map(image_bgr)

    output = []

    for detection in detections:

        bbox = detection["bbox"]

        distance_result = estimate_distance(
            depth_map,
            bbox,
            SCALE_FACTOR,
            OFFSET
        )

        output.append({
            "object": detection["object"],
            "confidence": detection["confidence"],
            "bbox": bbox,
            "raw_depth_m": distance_result["raw_depth_m"],
            "calibrated_distance_m": distance_result["calibrated_distance_m"]
        })

    return output

print("YOLO + Depth integration ready!")


In [ ]:

import cv2

def draw_detections(image_bgr, detections):

    output_image = image_bgr.copy()

    for item in detections:

        x1, y1, x2, y2 = item["bbox"]

        label = item["object"]
        distance = item["calibrated_distance_m"]

        text = f"{label}: {distance:.2f} m"

        cv2.rectangle(
            output_image,
            (x1, y1),
            (x2, y2),
            (0, 255, 0),
            2
        )

        cv2.putText(
            output_image,
            text,
            (x1, max(25, y1 - 10)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (0, 255, 0),
            2
        )

    return output_image

print("Bounding box visualization ready!")


In [ ]:

def process_frame(frame):
    """
    Process one frame received from the phone camera.
    Input: OpenCV BGR frame.
    Output: annotated frame and detections.
    """

    if frame is None:
        raise ValueError("Input frame is empty.")

    detections = detect_objects_with_distance(frame)

    output_frame = draw_detections(
        frame,
        detections
    )

    return output_frame, detections


print("process_frame is ready!")


In [ ]:
import time
import cv2

class RealTimeProcessor:

    def __init__(self):
        self.fps = 0.0
        self.alpha = 0.2
        self.total_frames = 0

    def process(self, frame):

        if frame is None:
            raise ValueError("Empty frame received")

        start_time = time.perf_counter()

        output_frame, detections = process_frame(frame)

        processing_time = time.perf_counter() - start_time

        self.total_frames += 1

        current_fps = (
            1.0 / processing_time
            if processing_time > 0 else 0
        )

        if self.fps == 0:
            self.fps = current_fps
        else:
            self.fps = (
                self.alpha * current_fps
                + (1 - self.alpha) * self.fps
            )

        cv2.putText(
            output_frame,
            f"Processing FPS: {self.fps:.2f}",
            (20, 35),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (0, 255, 255),
            2
        )

        return output_frame, detections, {
            "processing_time": processing_time,
            "processing_fps": self.fps,
            "total_frames": self.total_frames
        }


realtime_processor = RealTimeProcessor()

print("RealTimeProcessor is ready!")

## Real-time connection with the Android app
This notebook does **not** turn on the phone camera itself. The Flutter app opens the Android camera and continuously uploads frames to the API below. This notebook receives each frame, runs YOLO + Depth Anything + calibration, and returns detections, distance, annotated image, and FPS.
Run the cells from top to bottom. Keep this Colab session running while the app is connected. The public Cloudflare URL printed in the final cell is temporary and must be placed in the Flutter app by the teammate handling Flutter.


In [ ]:

!pip install -q fastapi uvicorn python-multipart nest-asyncio


In [ ]:
from ultralytics import YOLO

yolo_model = YOLO("yolo26n.pt")

exported_model = yolo_model.export(
    format="tflite",
    imgsz=320,
    batch=1
)

print("YOLO Android export completed!")
print("Exported model:", exported_model)

In [ ]:
import time
import threading

processing_lock = threading.Lock()

total_frames_processed = 0
total_processing_time = 0.0

def process_received_frame(frame):

    global total_frames_processed
    global total_processing_time

    if frame is None:
        raise ValueError("Empty frame received")

    if not processing_lock.acquire(blocking=False):
        return None

    try:
        start_time = time.perf_counter()

        # Process every received frame
        output_frame, detections = process_frame(frame)

        elapsed = time.perf_counter() - start_time

        total_frames_processed += 1
        total_processing_time += elapsed

        average_fps = (
            total_frames_processed / total_processing_time
            if total_processing_time > 0 else 0
        )

        processing_fps = (
            1 / elapsed if elapsed > 0 else 0
        )

        return {
            "output_frame": output_frame,
            "detections": detections,
            "processing_time": elapsed,
            "processing_fps": processing_fps,
            "average_fps": average_fps,
            "total_frames_processed": total_frames_processed
        }
    finally:
        processing_lock.release()

print("Continuous frame processing is ready!")

In [ ]:
!pip install -q onnx onnxruntime

In [ ]:
import torch
import onnx
import onnxruntime as ort
import numpy as np

print("All libraries imported successfully!")

In [ ]:
import torch
import onnx
import onnxruntime as ort
import numpy as np

# Prepare the loaded model
depth_model = depth_model.cpu().eval()

class ExportModel(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, image):
        return self.model(image)

export_model = ExportModel(depth_model).eval()

# Input dimensions
dummy_input = torch.randn(1, 3, 686, 518)

ONNX_PATH = "/content/depth_anything_metric.onnx"

with torch.no_grad():
    torch.onnx.export(
        export_model,
        dummy_input,
        ONNX_PATH,
        input_names=["image"],
        output_names=["depth"],
        opset_version=17,
        do_constant_folding=True,
        dynamo=False
    )

# Validate ONNX model
onnx_model = onnx.load(ONNX_PATH)
onnx.checker.check_model(onnx_model)

# Test inference
session = ort.InferenceSession(
    ONNX_PATH,
    providers=["CPUExecutionProvider"]
)

test_input = np.random.randn(
    1, 3, 686, 518
).astype(np.float32)

result = session.run(
    ["depth"],
    {"image": test_input}
)

print("ONNX export successful!")
print("Output shape:", result[0].shape)
print("Model size MB:", round(
    __import__("os").path.getsize(ONNX_PATH) / (1024 * 1024), 2
))

In [ ]:
# Compare PyTorch and ONNX outputs using the same input

test_input = np.random.randn(
    1, 3, 686, 518
).astype(np.float32)

with torch.no_grad():
    torch_result = depth_model(
        torch.from_numpy(test_input)
    ).cpu().numpy()

onnx_result = session.run(
    ["depth"],
    {"image": test_input}
)[0]

max_diff = np.max(
    np.abs(torch_result - onnx_result)
)

mean_diff = np.mean(
    np.abs(torch_result - onnx_result)
)

print("PyTorch shape:", torch_result.shape)
print("ONNX shape:", onnx_result.shape)
print("Maximum difference:", max_diff)
print("Mean difference:", mean_diff)

if np.allclose(
    torch_result,
    onnx_result,
    atol=1e-3,
    rtol=1e-3
):
    print("SUCCESS: ONNX matches PyTorch!")
else:
    print("WARNING: Outputs are different!")

# Restore the model to the selected runtime device for live frame processing.
depth_model = depth_model.to(device).eval()
print("Depth model restored to", device)


In [ ]:
import time
import threading

processing_lock = threading.Lock()
total_frames_processed = 0
total_processing_time = 0.0
realtime_processor = RealTimeProcessor()

def process_received_frame(frame):
    """Process one BGR frame received from the Android Flutter app."""
    global total_frames_processed, total_processing_time
    if frame is None or not isinstance(frame, np.ndarray):
        raise ValueError("Empty or invalid frame received")

    if not processing_lock.acquire(blocking=False):
        return None

    try:
        start_time = time.perf_counter()
        output_frame, detections, stats = realtime_processor.process(frame)
        elapsed = time.perf_counter() - start_time
        total_frames_processed += 1
        total_processing_time += elapsed
        average_fps = (total_frames_processed / total_processing_time
                       if total_processing_time > 0 else 0.0)

        # Convert NumPy/native values into JSON-safe Python values.
        safe_detections = []
        for item in detections:
            safe_detections.append({
                "object": str(item["object"]),
                "confidence": float(item["confidence"]),
                "bbox": [int(v) for v in item["bbox"]],
                "raw_depth_m": float(item["raw_depth_m"]),
                "calibrated_distance_m": float(item["calibrated_distance_m"])
            })

        return {
            "output_frame": output_frame,
            "detections": safe_detections,
            "processing_time": float(elapsed),
            "processing_fps": float(stats["processing_fps"]),
            "average_fps": float(average_fps),
            "total_frames_processed": int(total_frames_processed)
        }
    finally:
        processing_lock.release()

print("Frame receiver is ready")

In [ ]:
from fastapi import FastAPI, UploadFile, File, HTTPException
from fastapi.responses import JSONResponse
from fastapi.concurrency import run_in_threadpool
from fastapi.middleware.cors import CORSMiddleware
import base64
import cv2
import numpy as np

app = FastAPI(title="Basirati Real-Time Processing API")
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],  # Restrict to your app's origin/domain for production.
    allow_credentials=False,
    allow_methods=["GET", "POST"],
    allow_headers=["*"],
)

@app.get("/health")
async def health():
    return {"status": "ok", "message": "Basirati frame API is running"}

@app.post("/process-frame")
async def process_camera_frame(file: UploadFile = File(...)):
    image_bytes = await file.read()
    if not image_bytes:
        raise HTTPException(status_code=400, detail="Empty image upload")

    image_array = np.frombuffer(image_bytes, dtype=np.uint8)
    frame = cv2.imdecode(image_array, cv2.IMREAD_COLOR)
    if frame is None:
        raise HTTPException(status_code=400, detail="Invalid image frame")

    try:
        # Run the CPU/GPU-heavy model inference outside the async event loop.
        result = await run_in_threadpool(process_received_frame, frame)
        
        # If the lock is active, drop the frame to prevent API lag
        if result is None:
            return JSONResponse(
                status_code=200,
                content={"status": "dropped", "message": "Frame dropped to prevent lag"}
            )
            
        success, encoded = cv2.imencode(".jpg", result["output_frame"])
        if not success:
            raise ValueError("Could not encode processed frame")

        return {
            "status": "processed",
            "detections": result["detections"],
            "output_frame": base64.b64encode(encoded.tobytes()).decode("utf-8"),
            "processing_time": result["processing_time"],
            "processing_fps": result["processing_fps"],
            "average_fps": result["average_fps"],
            "total_frames_processed": result["total_frames_processed"]
        }
    except Exception as exc:
        raise HTTPException(status_code=500, detail=str(exc))

print("Basirati API routes are ready with anti-lag measures")

In [ ]:
# Start FastAPI and expose it through a temporary HTTPS URL for the Flutter app.
import os
import re
import subprocess
import threading
import time
import urllib.request
import uvicorn
import nest_asyncio

# Apply nest_asyncio to prevent event loop crashes in Colab
nest_asyncio.apply()

# Install Cloudflare Tunnel client in Colab (no account/token is normally needed).
if not os.path.exists("/usr/local/bin/cloudflared"):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
    !chmod +x /usr/local/bin/cloudflared

# Start the local API only once.
if "api_server_thread" not in globals() or not api_server_thread.is_alive():
    def _run_api():
        uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")
    api_server_thread = threading.Thread(target=_run_api, daemon=True)
    api_server_thread.start()
    time.sleep(3)

# Confirm that the local API is responding.
try:
    with urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=5) as response:
        print("Local API:", response.read().decode())
except Exception as exc:
    raise RuntimeError("FastAPI did not start. Check the previous cell output.") from exc

# Start a temporary HTTPS tunnel and extract its public URL.
if "cloudflared_process" not in globals() or cloudflared_process.poll() is not None:
    cloudflared_process = subprocess.Popen(
        ["/usr/local/bin/cloudflared", "tunnel", "--url", "http://127.0.0.1:8000", "--no-autoupdate"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )

public_url = None
start_wait = time.time()
while time.time() - start_wait < 60:
    line = cloudflared_process.stdout.readline()
    if not line and cloudflared_process.poll() is not None:
        break
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break

if not public_url:
    raise RuntimeError("Could not obtain Cloudflare URL. Re-run this cell and inspect Cloudflared output.")

print("\nAPI PUBLIC URL:", public_url)
print("Health check:", public_url + "/health")
print("Flutter frame endpoint:", public_url + "/process-frame")
print("Keep this Colab notebook running. The URL changes when the tunnel/runtime restarts.")